In [1]:
import pandas as pd
import skimage as ski
import mahotas
import numpy as np
import os
from pathlib import Path

In [ ]:
# The CSV just gotta have a img_id column
PATH = Path("../data/metadata.csv")
df = pd.read_csv(PATH)
df


,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,False,False,False,False,False,False,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,True,True,False,True,True,True,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,True,True,False,False,True,True,PAT_684_1302_588.png,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2293,PAT_1708,3156,NaN,NaN,NaN,NaN,73,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1708_3156_175.png,False
2294,PAT_46,880,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,12.0,BCC,True,True,False,True,False,False,PAT_46_880_140.png,True
2295,PAT_1343,1217,NaN,NaN,NaN,NaN,74,NaN,NaN,NaN,...,NaN,SEK,False,False,False,False,False,False,PAT_1343_1217_404.png,False
2296,PAT_326,690,False,False,POMERANIA,POMERANIA,58,True,FEMALE,True,...,4.0,BCC,True,False,False,False,False,True,PAT_326_690_823.png,True


In [3]:
# Filter away maskless images
mask_dir = Path("../data/masks")

imgID = df["img_id"].to_list()

maskExists = []

for i in imgID:
    base_name = Path(i).name
    name = Path(base_name).stem 
    ext = Path(base_name).suffix  
    
    mask_path = mask_dir / f"{name}_mask{ext}"
    maskExists.append(mask_path.exists())

In [4]:
df = df[maskExists]
df

,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,False,False,False,False,False,False,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,True,True,False,True,True,True,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,True,True,False,False,True,True,PAT_684_1302_588.png,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2289,PAT_386,785,True,False,POMERANIA,POMERANIA,66,False,MALE,True,...,15.0,ACK,True,False,True,False,True,True,PAT_386_785_536.png,True
2291,PAT_273,421,False,False,POMERANIA,POMERANIA,41,True,MALE,False,...,5.0,BCC,True,UNK,True,UNK,True,True,PAT_273_421_905.png,True
2292,PAT_491,934,False,False,POMERANIA,POMERANIA,43,True,FEMALE,True,...,5.0,SCC,True,UNK,False,UNK,True,True,PAT_491_934_46.png,True
2294,PAT_46,880,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,12.0,BCC,True,True,False,True,False,False,PAT_46_880_140.png,True


The following will take a small while

In [5]:
img_dir = Path("../data/imgs")
mask_dir = Path("../data/masks")

imgID = df["img_id"].to_list()
diagnosis = df["diagnostic"].to_list()
compactnesses = []
darkCos = []

for i in range(len(imgID)):
    img_path = img_dir / imgID[i]
    mask_path = mask_dir / f"{img_path.stem}_mask{img_path.suffix}"

    img = ski.io.imread(img_path)
    img = ski.transform.resize(img, (255, 255))

    mask = ski.io.imread(mask_path)
    mask = ski.transform.resize(mask, (255, 255), preserve_range=True)

    # mask to boolean
    if mask.ndim == 3:
        mask = ski.color.rgb2gray(mask) > 0.5
    else:
        mask = mask > 0.5

    #compactness
    area = mask.sum()
    perimeter = area - ski.morphology.erosion(mask,ski.morphology.disk(3)).sum()  # erosion can be refined to be even finer
    compactness = perimeter**2 / (area * 12)
    compactnesses.append(compactness)

    #darkness coefficient
    # takes how many pixels are light vs dark
    darkCo = (img[mask][:, :3].sum(axis=1)/3 > 0.5).sum() / img[mask].shape[0]
    darkCos.append(darkCo)

# Add to dataframe
df["compactness"] = compactnesses
df["dark"] = darkCos

C:\Users\johan\AppData\Local\Temp\ipykernel_9404\1376886686.py:28: RuntimeWarning: invalid value encountered in scalar divide
  compactness = perimeter**2 / (area * 12)
C:\Users\johan\AppData\Local\Temp\ipykernel_9404\1376886686.py:33: RuntimeWarning: invalid value encountered in scalar divide
  darkCo = (img[mask][:, :3].sum(axis=1)/3 > 0.5).sum() / img[mask].shape[0]
C:\Users\johan\AppData\Local\Temp\ipykernel_9404\1376886686.py:28: RuntimeWarning: invalid value encountered in scalar divide
  compactness = perimeter**2 / (area * 12)
C:\Users\johan\AppData\Local\Temp\ipykernel_9404\1376886686.py:33: RuntimeWarning: invalid value encountered in scalar divide
  darkCo = (img[mask][:, :3].sum(axis=1)/3 > 0.5).sum() / img[mask].shape[0]
C:\Users\johan\AppData\Local\Temp\ipykernel_9404\1376886686.py:28: RuntimeWarning: invalid value encountered in scalar divide
  compactness = perimeter**2 / (area * 12)
C:\Users\johan\AppData\Local\Temp\ipykernel_9404\1376886686.py:33: RuntimeWarning: inva

In [7]:
df

,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed,compactness,dark
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,False,False,False,False,False,False,PAT_1516_1765_530.png,False,9.315380,0.003915
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,True,True,False,True,True,True,PAT_46_881_939.png,True,12.404898,0.834343
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,True,False,False,False,False,False,PAT_1545_1867_547.png,False,11.676342,0.097100
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,True,False,False,False,False,False,PAT_1989_4061_934.png,False,8.022908,0.716690
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,True,True,False,False,True,True,PAT_684_1302_588.png,True,8.629233,0.025155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2289,PAT_386,785,True,False,POMERANIA,POMERANIA,66,False,MALE,True,...,True,False,True,False,True,True,PAT_386_785_536.png,True,8.702078,0.337758
2291,PAT_273,421,False,False,POMERANIA,POMERANIA,41,True,MALE,False,...,True,UNK,True,UNK,True,True,PAT_273_421_905.png,True,8.683203,0.072259
2292,PAT_491,934,False,False,POMERANIA,POMERANIA,43,True,FEMALE,True,...,True,UNK,False,UNK,True,True,PAT_491_934_46.png,True,8.650502,0.928517
2294,PAT_46,880,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,True,True,False,True,False,False,PAT_46_880_140.png,True,9.134379,0.403239


In [6]:
#save as feature extracted CSV
PATH = Path("../data/metadataImgFeat.csv")
df.to_csv(PATH)